In [59]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
#

In [60]:
df_data_sus = pd.read_csv("DADOS.csv")

In [61]:
df_IDEB_inicio = pd.read_csv("IDEB_inicio.csv")
df_IDEB_final = pd.read_csv("IDEB_final.csv")
df_escolarizacao = pd.read_csv("taxa_escolarizacao.csv")

In [62]:
df_populacao = pd.read_csv("populacao.csv")
df_densidade_demografica = pd.read_csv("densidade_demo.csv")

## Tratamento da base de dados SUS

In [63]:
df_data_sus.head()

,ID,MUNICÍPIO,PRIMEIRO_NOME
0,1,ABAIARA ...,ADRIANA
1,2,ABAIARA ...,ADRIANA
2,3,ABAIARA ...,ALINE
3,4,ABAIARA ...,ALZENIR
4,5,ABAIARA ...,ANA


In [64]:
df_data_sus['MUNICÍPIO'] = df_data_sus['MUNICÍPIO'].str.strip()
df_data_sus['PRIMEIRO_NOME'] = df_data_sus['PRIMEIRO_NOME'].str.strip()

In [65]:
df_data_sus['MUNICÍPIO'] = df_data_sus['MUNICÍPIO'].astype('category')

In [66]:
df_data_sus.set_index('ID', inplace=True)

In [67]:
# 1. Remover registros onde o Município é nulo (NaN)
df_data_sus = df_data_sus.dropna(subset=['MUNICÍPIO'])

In [68]:
# Remover acentos e cedilhas (ex: de 'VIÇOSA' para 'VICOSA')
# O normalize('NFKD') separa a letra do acento, e o encode/decode descarta o acento
df_data_sus['Local'] = (
    df_data_sus['MUNICÍPIO']
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

In [69]:
df_casos_sus = df_data_sus.groupby('Local').size().reset_index(name='TOTAL_REGISTROS')

In [70]:
df_casos_sus.head()

,Local,TOTAL_REGISTROS
0,ABAIARA,231
1,ACARAPE,495
2,ACARAU,2368
3,ACOPIARA,1331
4,AIUABA,570


## Tratametno IDEB

In [71]:
df_IDEB_inicio.head()

,Local,"""IDEB – Anos iniciais do ensino fundamental (Rede pública)"""
0,Jucás,7.0
1,Senador Sá,7.3
2,Morada Nova,6.0
3,Chaval,6.5
4,Pentecoste,6.7


In [72]:
df_IDEB_final.head()

,Local,"""IDEB – Anos finais do ensino fundamental (Rede pública)"""
0,Jucás,5.8
1,Senador Sá,6.2
2,Morada Nova,5.4
3,Chaval,5.2
4,Pentecoste,5.6


In [73]:
df_IDEB_medio = pd.merge(df_IDEB_inicio, df_IDEB_final, on='Local')


In [74]:
df_IDEB_medio.columns

Index(['Local', ' "IDEB – Anos iniciais do ensino fundamental (Rede pública)"',
       ' "IDEB – Anos finais do ensino fundamental (Rede pública)"'],
      dtype='str')

In [75]:
df_IDEB_medio['IDEB_medio'] = (df_IDEB_medio[' "IDEB – Anos iniciais do ensino fundamental (Rede pública)"'] + df_IDEB_medio[' "IDEB – Anos finais do ensino fundamental (Rede pública)"']) / 2

In [76]:
df_IDEB_medio.head()

,Local,"""IDEB – Anos iniciais do ensino fundamental (Rede pública)""","""IDEB – Anos finais do ensino fundamental (Rede pública)""",IDEB_medio
0,Jucás,7.0,5.8,6.40
1,Senador Sá,7.3,6.2,6.75
2,Morada Nova,6.0,5.4,5.70
3,Chaval,6.5,5.2,5.85
4,Pentecoste,6.7,5.6,6.15


In [77]:
df_IDEB = df_IDEB_medio[['Local', 'IDEB_medio']]

## Tratamento taxa de escolarização

In [78]:
df_escolarizacao.info()

<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 2 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Local                                             184 non-null    str    
 1    "Taxa de escolarização de 6 a 14 anos de idade"  184 non-null    float64
dtypes: float64(1), str(1)
memory usage: 3.0 KB


In [79]:
df_escolarizacao.head()

,Local,"""Taxa de escolarização de 6 a 14 anos de idade"""
0,Jucás,97.74
1,Senador Sá,99.10
2,Morada Nova,98.55
3,Chaval,100.00
4,Pentecoste,98.42


In [80]:
df_escolarizacao.columns

Index(['Local', ' "Taxa de escolarização de 6 a 14 anos de idade"'], dtype='str')

In [81]:
df_escolarizacao["Taxa de escolarização de 6 a 14 anos de idade"] = df_escolarizacao[' "Taxa de escolarização de 6 a 14 anos de idade"']
df_escolariazacao = df_escolarizacao[['Local', 'Taxa de escolarização de 6 a 14 anos de idade']]

In [82]:
df_escolariazacao.head()

,Local,Taxa de escolarização de 6 a 14 anos de idade
0,Jucás,97.74
1,Senador Sá,99.10
2,Morada Nova,98.55
3,Chaval,100.00
4,Pentecoste,98.42


In [83]:
df_escolariazacao.columns

Index(['Local', 'Taxa de escolarização de 6 a 14 anos de idade'], dtype='str')

## Tratamento população

In [84]:
df_populacao.head()

,Local,"""População no último censo"""
0,Jucás,23922
1,Senador Sá,7262
2,Morada Nova,61443
3,Chaval,12462
4,Pentecoste,37813


In [85]:
df_populacao.columns

Index(['Local', ' "População no último censo"'], dtype='str')

In [86]:
df_populacao.rename(columns={' "População no último censo"': 'Populacao'}, inplace=True)

## Tratamento Densidade Demografica

In [87]:
df_densidade_demografica.head()

,Local,"""Densidade demográfica"""
0,Jucás,25.44
1,Senador Sá,17.10
2,Morada Nova,22.23
3,Chaval,52.53
4,Pentecoste,27.40


In [88]:
df_densidade_demografica.columns

Index(['Local', ' "Densidade demográfica"'], dtype='str')

In [89]:
df_densidade_demografica.rename(columns={' "Densidade demográfica"': 'Densidade Demografica'}, inplace=True)

## União dos df

In [90]:
import unicodedata

In [91]:
df_ibge = pd.merge(df_IDEB, df_escolarizacao, on='Local', how='outer')

In [92]:
def limpar_texto(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return texto

In [93]:
df_ibge['Local'] = df_ibge['Local'].apply(limpar_texto)
df_populacao['Local'] = df_populacao['Local'].apply(limpar_texto)
df_casos_sus['Local'] = df_casos_sus['Local'].apply(limpar_texto)
df_densidade_demografica['Local'] = df_densidade_demografica['Local'].apply(limpar_texto)

In [94]:
df_ibge.head()

,Local,IDEB_medio,"""Taxa de escolarização de 6 a 14 anos de idade""",Taxa de escolarização de 6 a 14 anos de idade
0,ABAIARA,5.75,99.69,99.69
1,ACARAPE,5.60,97.64,97.64
2,ACARAU,5.90,99.45,99.45
3,ACOPIARA,5.40,99.45,99.45
4,AIUABA,6.00,98.87,98.87


In [95]:
df_semifinal = pd.merge(df_casos_sus, df_ibge, left_on='Local', right_on='Local', how='left')

In [96]:
df_semifinal2 = pd.merge(df_semifinal, df_populacao, left_on='Local', right_on='Local', how='left')

In [97]:
df_final = pd.merge(df_semifinal2, df_densidade_demografica, left_on='Local', right_on='Local', how='left')

In [98]:
df_final.head()

,Local,TOTAL_REGISTROS,IDEB_medio,"""Taxa de escolarização de 6 a 14 anos de idade""",Taxa de escolarização de 6 a 14 anos de idade,Populacao,Densidade Demografica
0,ABAIARA,231,5.75,99.69,99.69,10038.0,55.51
1,ACARAPE,495,5.60,97.64,97.64,14027.0,107.90
2,ACARAU,2368,5.90,99.45,99.45,65264.0,77.47
3,ACOPIARA,1331,5.40,99.45,99.45,44962.0,19.95
4,AIUABA,570,6.00,98.87,98.87,14076.0,5.77


In [99]:
df_final.columns

Index(['Local', 'TOTAL_REGISTROS', 'IDEB_medio',
       ' "Taxa de escolarização de 6 a 14 anos de idade"',
       'Taxa de escolarização de 6 a 14 anos de idade', 'Populacao',
       'Densidade Demografica'],
      dtype='str')

In [100]:
df_final = df_final.loc[:, ~df_final.columns.duplicated()]
df_final.rename(columns={'Taxa de escolarização de 6 a 14 anos de idade': 'TAXA_ESCOLARIZACAO'}, inplace=True)

In [101]:
# Cria uma coluna booleana (True se tiver mais atendimento que habitante, False caso contrário)
df_final['ALERTA_ATENDIMENTO_ALTO'] = df_final['TOTAL_REGISTROS'] > df_final['Densidade Demografica']

# Para ver quais cidades caíram nesse alerta:
cidades_alerta = df_final[df_final['ALERTA_ATENDIMENTO_ALTO'] == True]
print("Cidades com mais atendimentos que habitantes:")
print(cidades_alerta[['Local', 'TOTAL_REGISTROS', 'Densidade Demografica']])

Cidades com mais atendimentos que habitantes:
               Local  TOTAL_REGISTROS  Densidade Demografica
0            ABAIARA              231                  55.51
1            ACARAPE              495                 107.90
2             ACARAU             2368                  77.47
3           ACOPIARA             1331                  19.95
4             AIUABA              570                   5.77
..               ...              ...                    ...
179      URUBURETAMA             1524                 203.11
180           URUOCA              735                  19.70
181          VARJOTA              675                 101.01
182    VARZEA ALEGRE              874                  46.97
183  VICOSA DO CEARA             2899                  45.55

[182 rows x 3 columns]


In [102]:
# 1. Remover a coluna duplicada (a que tem aspas no nome, se existir)
if '"Taxa de escolarização de 6 a 14 anos de idade"' in df_final.columns:
    df_final = df_final.drop(columns=['"Taxa de escolarização de 6 a 14 anos de idade"'])

# 2. Renomear para nomes limpos e padronizados (Melhor prática de BI)
df_final.rename(columns={
    'Taxa de escolarização de 6 a 14 anos de idade': 'TAXA_ESCOLARIZACAO',
    'Densidade demográfica': 'DENSIDADE_DEMOGRAFICA',
    'Local': 'MUNICÍPIO' # Para garantir que a chave tem o nome correto
}, inplace=True)

# Visualizar como ficou limpo
# 1. Validar coerência: Há municípios com mais atendimentos que habitantes?
df_final['ALERTA_MAIS_ATENDIMENTOS'] = df_final['TOTAL_REGISTROS'] > df_final['Populacao']

# 2. Criar a métrica estrela do BI: Proporção de atendimentos por 100 mil habitantes
df_final['ATENDIMENTOS_POR_100K'] = (df_final['TOTAL_REGISTROS'] / df_final['Populacao']) * 100000

# Arredondar para ficar apresentável
df_final['ATENDIMENTOS_POR_100K'] = df_final['ATENDIMENTOS_POR_100K'].round(2)
df_final.head()

,MUNICÍPIO,TOTAL_REGISTROS,IDEB_medio,"""Taxa de escolarização de 6 a 14 anos de idade""",TAXA_ESCOLARIZACAO,Populacao,Densidade Demografica,ALERTA_ATENDIMENTO_ALTO,ALERTA_MAIS_ATENDIMENTOS,ATENDIMENTOS_POR_100K
0,ABAIARA,231,5.75,99.69,99.69,10038.0,55.51,True,False,2301.26
1,ACARAPE,495,5.60,97.64,97.64,14027.0,107.90,True,False,3528.91
2,ACARAU,2368,5.90,99.45,99.45,65264.0,77.47,True,False,3628.34
3,ACOPIARA,1331,5.40,99.45,99.45,44962.0,19.95,True,False,2960.28
4,AIUABA,570,6.00,98.87,98.87,14076.0,5.77,True,False,4049.45


In [103]:
# 1. Remover as colunas duplicadas ou antigas
colunas_para_remover = [
    '"Taxa de escolarização de 6 a 14 anos de idade"',
    'ALERTA_ATENDIMENTO_ALTO' # Mantemos apenas o ALERTA_MAIS_ATENDIMENTOS
]
df_final = df_final.drop(columns=colunas_para_remover, errors='ignore')

# 2. Exportar para CSV (A base de ouro)
df_final.to_csv('BASE_FINAL_BI_SUS.csv', index=False, encoding='utf-8-sig')

print("Base exportada com sucesso! Pronta para o BI.")

Base exportada com sucesso! Pronta para o BI.


In [104]:
df_final.head()

,MUNICÍPIO,TOTAL_REGISTROS,IDEB_medio,"""Taxa de escolarização de 6 a 14 anos de idade""",TAXA_ESCOLARIZACAO,Populacao,Densidade Demografica,ALERTA_MAIS_ATENDIMENTOS,ATENDIMENTOS_POR_100K
0,ABAIARA,231,5.75,99.69,99.69,10038.0,55.51,False,2301.26
1,ACARAPE,495,5.60,97.64,97.64,14027.0,107.90,False,3528.91
2,ACARAU,2368,5.90,99.45,99.45,65264.0,77.47,False,3628.34
3,ACOPIARA,1331,5.40,99.45,99.45,44962.0,19.95,False,2960.28
4,AIUABA,570,6.00,98.87,98.87,14076.0,5.77,False,4049.45


## Adicionando as coordenadas

In [108]:
# 1. Carregar uma base pública confiável com as coordenadas do Brasil
url_coords = "https://raw.githubusercontent.com/kelvins/Municipios-Brasileiros/main/csv/municipios.csv"
df_coords = pd.read_csv(url_coords)

# 2. Filtrar apenas os municípios do Ceará (O código UF do Ceará é 23)
df_coords_ce = df_coords[df_coords['codigo_uf'] == 23].copy()

# 3. Aplicar a SUA função de limpeza para o nome da cidade ficar igualzinho à base do SUS
# (Certifique-se de que a função limpar_texto que criamos antes já foi executada)
df_coords_ce['MUNICÍPIO'] = df_coords_ce['nome'].apply(limpar_texto)

# 4. Selecionar apenas as colunas que importam
df_coords_ce = df_coords_ce[['MUNICÍPIO', 'latitude', 'longitude']]

# 5. Fazer o cruzamento com o seu DataFrame final atual
df_final = pd.merge(df_final, df_coords_ce, on='MUNICÍPIO', how='left')

# 6. Renomear para maiúsculo, para o Streamlit reconhecer automaticamente
df_final.rename(columns={'latitude': 'LATITUDE', 'longitude': 'LONGITUDE'}, inplace=True)

# Visualizar se deu certo
print(df_final[['MUNICÍPIO', 'LATITUDE', 'LONGITUDE']].head())

# 7. Exportar novamente a base final (Agora com coordenadas!)
df_final.to_csv('BASE_FINAL_BI_SUS.csv', index=False, encoding='utf-8-sig')

  MUNICÍPIO  LATITUDE  LONGITUDE
0   ABAIARA  -7.34588   -39.0416
1   ACARAPE  -4.22083   -38.7055
2    ACARAU  -2.88769   -40.1183
3  ACOPIARA  -6.08911   -39.4480
4    AIUABA  -6.57122   -40.1178


## Metricas

### Métricas Essenciais (O básico bem feito):

- Total de Atendimentos Analisados vs. População Total Alcançada: Mostra o volume do projeto.

- Taxa Média Estadual (Atendimentos por 100k hab.): Essa será a sua "linha de base". Tudo será comparado para saber se está acima ou abaixo dessa média.

- Índice de Sobrecarga (O Alerta): O número exato ou percentual de municípios onde a demanda superou a população residente (ex: "X% dos municípios cearenses atendem mais pessoas do que sua própria população").

Se olharmos para os seus números totais, a sua base do SUS (DADOS.txt) tem 204.579 registros. Já a população total do Ceará mapeada é de 8.748.531 habitantes.

Isso significa que o seu arquivo de dados representa uma amostra onde, na média estadual, apenas cerca de 2,3% da população utilizou o SUS (provavelmente essa base de dados é referente a um único mês, a uma especialidade médica específica, ou é uma amostra reduzida para fins de estudo).

In [109]:
# --- CÉLULA 1: AUDITORIA DE TOTAIS ---
total_atendimentos = df_final['TOTAL_REGISTROS'].sum()
populacao_total = df_final['Populacao'].sum()
taxa_media_estadual = (total_atendimentos / populacao_total) * 100000

print("=== 1. AUDITORIA MACRO ===")
print(f"Total de Registos SUS: {total_atendimentos:,.0f}")
print(f"População Total IBGE: {populacao_total:,.0f}")
print(f"Taxa Média do Estado: {taxa_media_estadual:.2f} atendimentos a cada 100 mil habitantes")

=== 1. AUDITORIA MACRO ===
Total de Registos SUS: 204,579
População Total IBGE: 8,748,531
Taxa Média do Estado: 2338.44 atendimentos a cada 100 mil habitantes


In [110]:
# --- CÉLULA 2: VALIDAÇÃO DE COERÊNCIA ---
# Filtra as cidades onde os registos superam a população
cidades_alerta = df_final[df_final['TOTAL_REGISTROS'] > df_final['Populacao']]
qtd_alerta = len(cidades_alerta)

print("=== 2. TESTE DE COERÊNCIA ===")
if qtd_alerta == 0:
    print("✅ SUCESSO: Nenhum município registou mais atendimentos do que a própria população.")
else:
    print(f"⚠️ ALERTA: {qtd_alerta} municípios com volume suspeito/sobrecarregado.")
    print(cidades_alerta[['MUNICÍPIO', 'TOTAL_REGISTROS', 'Populacao']])

=== 2. TESTE DE COERÊNCIA ===
✅ SUCESSO: Nenhum município registou mais atendimentos do que a própria população.


In [111]:
# --- CÉLULA 3: ÍNDICE DE SOBRECARGA ---
# Conta quantas cidades ultrapassaram a taxa média do estado
cidades_sobrecarregadas = df_final[df_final['ATENDIMENTOS_POR_100K'] > taxa_media_estadual]
qtd_sobrecarga = len(cidades_sobrecarregadas)
percentual_sobrecarga = (qtd_sobrecarga / len(df_final)) * 100

print("=== 3. ÍNDICE DE SOBRECARGA ===")
print(f"Municípios acima da média estadual: {qtd_sobrecarga} de {len(df_final)}")
print(f"Percentagem de Sobrecarga: {percentual_sobrecarga:.2f}%")

# Para ver quais são as top 5 mais sobrecarregadas nesse filtro:
print("\nTop 5 Cidades mais acima da média:")
print(cidades_sobrecarregadas[['MUNICÍPIO', 'ATENDIMENTOS_POR_100K']].sort_values(by='ATENDIMENTOS_POR_100K', ascending=False).head(5))

=== 3. ÍNDICE DE SOBRECARGA ===
Municípios acima da média estadual: 135 de 184
Percentagem de Sobrecarga: 73.37%

Top 5 Cidades mais acima da média:
       MUNICÍPIO  ATENDIMENTOS_POR_100K
137    PENAFORTE                8392.78
172    TEJUCUOCA                7829.08
90       ITATIRA                7789.86
156      SALITRE                7695.55
179  URUBURETAMA                7548.67


In [112]:
# --- CÉLULA 4: AUDITORIA DOS RANKINGS EXTREMOS ---
print("=== 4. RANKINGS DE PROPORÇÃO (Por 100k hab.) ===")

print("\n🔴 TOP 5 MAIORES PROPORÇÕES (Gargalos):")
top_5 = df_final.nlargest(5, 'ATENDIMENTOS_POR_100K')
print(top_5[['MUNICÍPIO', 'ATENDIMENTOS_POR_100K', 'TOTAL_REGISTROS', 'Populacao']])

print("\n🟢 TOP 5 MENORES PROPORÇÕES (Estáveis / Subnotificados):")
bottom_5 = df_final.nsmallest(5, 'ATENDIMENTOS_POR_100K')
print(bottom_5[['MUNICÍPIO', 'ATENDIMENTOS_POR_100K', 'TOTAL_REGISTROS', 'Populacao']])

=== 4. RANKINGS DE PROPORÇÃO (Por 100k hab.) ===

🔴 TOP 5 MAIORES PROPORÇÕES (Gargalos):
       MUNICÍPIO  ATENDIMENTOS_POR_100K  TOTAL_REGISTROS  Populacao
137    PENAFORTE                8392.78              753     8972.0
172    TEJUCUOCA                7829.08             1343    17154.0
90       ITATIRA                7789.86             1591    20424.0
156      SALITRE                7695.55             1280    16633.0
179  URUBURETAMA                7548.67             1524    20189.0

🟢 TOP 5 MENORES PROPORÇÕES (Estáveis / Subnotificados):
    MUNICÍPIO  ATENDIMENTOS_POR_100K  TOTAL_REGISTROS  Populacao
58  FORTALEZA                 385.55             9364  2428708.0
54      ERERE                 602.41               39     6474.0
55    EUSEBIO                 690.31              512    74170.0
43    CAUCAIA                 823.78             2930   355679.0
70  HORIZONTE                 836.06              625    74755.0


In [113]:
# --- CÉLULA 5: IMPACTO DA TAXA DE ESCOLARIZAÇÃO ---
print("=== 5. IMPACTO DA ESCOLARIZAÇÃO NA DEMANDA SUS ===")

# Encontrar a mediana da taxa de escolarização
mediana_escolarizacao = df_final['TAXA_ESCOLARIZACAO'].median()

# Classificar as cidades
df_final['PERFIL_ESCOLARIZACAO'] = np.where(df_final['TAXA_ESCOLARIZACAO'] >= mediana_escolarizacao, 'Alta Escolarização', 'Baixa Escolarização')

# Calcular a média de atendimentos do SUS para cada grupo
impacto_escolarizacao = df_final.groupby('PERFIL_ESCOLARIZACAO')['ATENDIMENTOS_POR_100K'].mean().round(2)

print(f"Mediana de Escolarização do Ceará: {mediana_escolarizacao}%")
print("\nMédia de Atendimentos SUS (por 100k hab.):")
print(impacto_escolarizacao)

=== 5. IMPACTO DA ESCOLARIZAÇÃO NA DEMANDA SUS ===
Mediana de Escolarização do Ceará: 99.09%

Média de Atendimentos SUS (por 100k hab.):
PERFIL_ESCOLARIZACAO
Alta Escolarização     3621.50
Baixa Escolarização    3642.67
Name: ATENDIMENTOS_POR_100K, dtype: float64


In [114]:
# --- CÉLULA 6: CORRELAÇÃO ESTATÍSTICA (PEARSON) ---
print("=== 6. PROVA ESTATÍSTICA (CORRELAÇÃO) ===")

# Calcula a correlação entre a educação e a taxa de atendimentos
correlacao_ideb = df_final['IDEB_medio'].corr(df_final['ATENDIMENTOS_POR_100K'])
correlacao_esc = df_final['TAXA_ESCOLARIZACAO'].corr(df_final['ATENDIMENTOS_POR_100K'])

print(f"Correlação IDEB x SUS: {correlacao_ideb:.3f}")
print(f"Correlação Escolarização x SUS: {correlacao_esc:.3f}")

print("\nComo ler isso:")
print("- Valores próximos de 0: Não há relação.")
print("- Valores negativos: Relação inversa (Quando a Educação MELHORA, a demanda do SUS CAI).")
print("- Valores positivos: Relação direta (Quando um sobe, o outro também sobe).")

=== 6. PROVA ESTATÍSTICA (CORRELAÇÃO) ===
Correlação IDEB x SUS: -0.085
Correlação Escolarização x SUS: -0.006

Como ler isso:
- Valores próximos de 0: Não há relação.
- Valores negativos: Relação inversa (Quando a Educação MELHORA, a demanda do SUS CAI).
- Valores positivos: Relação direta (Quando um sobe, o outro também sobe).


In [115]:
# --- CÉLULA 7: CIDADES NO QUADRANTE DE EXCELÊNCIA ---
print("=== 7. CIDADES MODELO (ALTA EDUCAÇÃO + BAIXA DEMANDA SUS) ===")

# Define o que é "IDEB Alto" (Top 25% maiores notas)
ideb_excelente = df_final['IDEB_medio'] >= df_final['IDEB_medio'].quantile(0.75)

# Define o que é "Demanda Baixa" (Top 25% menores taxas de atendimento)
demanda_baixa = df_final['ATENDIMENTOS_POR_100K'] <= df_final['ATENDIMENTOS_POR_100K'].quantile(0.25)

# Filtra as cidades que cumprem os DOIS requisitos
cidades_modelo = df_final[ideb_excelente & demanda_baixa]

print(f"Encontramos {len(cidades_modelo)} municípios no Quadrante de Excelência.")
if len(cidades_modelo) > 0:
    print("\nLista de Cidades Modelo:")
    print(cidades_modelo[['MUNICÍPIO', 'IDEB_medio', 'TAXA_ESCOLARIZACAO', 'ATENDIMENTOS_POR_100K']].sort_values(by='IDEB_medio', ascending=False))

=== 7. CIDADES MODELO (ALTA EDUCAÇÃO + BAIXA DEMANDA SUS) ===
Encontramos 14 municípios no Quadrante de Excelência.

Lista de Cidades Modelo:
                     MUNICÍPIO  IDEB_medio  TAXA_ESCOLARIZACAO  \
53   DEPUTADO IRAPUAN PINHEIRO        9.10               98.51   
14                    ARARENDA        8.85               98.31   
166                     SOBRAL        8.75               98.98   
52                        CRUZ        8.00               99.57   
78                  IPAPORANGA        7.70               98.05   
65                    GROAIRAS        7.10               99.27   
81                    IPUEIRAS        6.70               96.94   
60                FRECHEIRINHA        6.65               98.28   
59                      FORTIM        6.55               99.23   
84                    ITAICABA        6.55               99.70   
62                       GRACA        6.50               98.94   
55                     EUSEBIO        6.50               99.34   
